# 🐦 Twitter (X) Data Crawler Sederhana untuk Google Colab

Notebook ini menggunakan **`tweet-harvest`** untuk mengambil data tweet berdasarkan **kata kunci** dan **periode waktu (tanggal mulai & akhir)**, lalu memuatnya ke dalam **Pandas DataFrame** dan mengekspornya ke file **CSV**.

> **Catatan Penting:** Sejak perubahan kebijakan X/Twitter, pengambilan data mewajibkan `auth_token` dari akun X/Twitter Anda (disarankan menggunakan akun cadangan/dummy).

### 1. Instalasi Dependencies (Jalankan Sekali di Awal)

In [ ]:
# Install Node.js & dependencies yang dibutuhkan oleh tweet-harvest di Google Colab
!apt-get update -qq
!apt-get install -y -qq ca-certificates curl gnupg
!mkdir -p /etc/apt/keyrings
!curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg
!echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_20.x nodistro main" | tee /etc/apt/sources.list.d/nodesource.list
!apt-get update -qq
!apt-get install -y -qq nodejs libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2
!npx playwright install chromium

### 2. Input Parameter & Jalankan Crawling
**Cara mendapatkan `twitter_auth_token`:**
1. Buka [x.com](https://x.com) di browser dan login.
2. Tekan `F12` (Inspect Element) -> buka tab **Application** (Chrome/Edge) atau **Storage** (Firefox).
3. Pilih **Cookies** -> `https://x.com`.
4. Cari cookie bernama **`auth_token`**, lalu salin nilainya (value) ke bawah.

In [ ]:
import os
import pandas as pd

# ==================== KONFIGURASI INPUT ====================
twitter_auth_token = "ISI_AUTH_TOKEN_ANDA_DI_SINI"  # @param {type:"string"}
kata_kunci = "timnas indonesia"                     # @param {type:"string"}
tanggal_mulai = "2024-01-01"                        # @param {type:"string"} (Format: YYYY-MM-DD)
tanggal_akhir = "2024-12-31"                        # @param {type:"string"} (Format: YYYY-MM-DD)
bahasa = "id"                                       # @param {type:"string"} (id = Indonesia, en = Inggris, kosongkan jika semua bahasa)
jumlah_maksimal_tweet = 100                         # @param {type:"integer"}
nama_file_csv = "hasil_crawling_twitter.csv"        # @param {type:"string"}
# ===========================================================

# Susun query pencarian dengan filter tanggal
search_query = f"{kata_kunci} since:{tanggal_mulai} until:{tanggal_akhir}"
if bahasa.strip():
    search_query += f" lang:{bahasa.strip()}"

print(f"🔍 Query Pencarian: {search_query}")
print(f"⏳ Memulai proses crawling hingga {jumlah_maksimal_tweet} tweet...")

# Jalankan tweet-harvest
!npx -y tweet-harvest@2.6.1 -o "{nama_file_csv}" -s "{search_query}" --tab "LATEST" -l {jumlah_maksimal_tweet} --token {twitter_auth_token}

### 3. Muat ke DataFrame & Unduh CSV

In [ ]:
# tweet-harvest menyimpan hasil di dalam folder 'tweets-data/'
file_path = f"tweets-data/{nama_file_csv}"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Berhasil mengambil {len(df)} baris data tweet!")
    
    # Simpan juga di root folder Colab agar mudah diakses
    df.to_csv(nama_file_csv, index=False, encoding="utf-8-sig")
    print(f"💾 File CSV tersimpan di: {nama_file_csv}")
    
    # Tampilkan 5 data teratas
    display(df.head())
    
    # Otomatis unduh file CSV jika dijalankan di Google Colab
    try:
        from google.colab import files
        files.download(nama_file_csv)
    except ImportError:
        pass
else:
    print("❌ File hasil crawling tidak ditemukan. Pastikan 'twitter_auth_token' valid dan query memiliki hasil.")